# Outlier detection methods (Catalog 2)

Catalog 2 twin of `02_methods.ipynb`. Runs the same three outlier detection methods on
the cleaned Catalog 2 features. The three come from three different principles so we are
not just measuring the same idea three times:

- **Robust Mahalanobis (MCD)**, distance from a clean centre.
- **Local Outlier Factor (LOF)**, density relative to local neighbours, swept over k.
- **Extended Isolation Forest (EIF)**, how easy a point is to isolate by random
  hyperplane splits.

Each method writes a score and a rank per burst. The validation notebook then checks
whether known repeaters concentrate at the top of those rankings.

Differences from the Cat 1 run, all flagged where they occur:
- Eight features, not nine (`scat_time` dropped in cleaning; see that notebook's header).
- ~4,500 rows instead of 558, so the LOF k-sweep is re-bracketed around the new
  sqrt(N) (~67) to keep 'local' meaning the same fraction of the data as it did for
  Cat 1. (2026-06-07 decision; revert to Cat 1's {10,20,30,50,75} if you want literal
  comparability instead.)
- Identity is `tns_name` + `sub_num` (tns_name repeats across sub-components), and
  one-offs are marked by a blank `repeater_name` rather than Cat 1's `-9999`.

## Setup: load the cleaned features

In [1]:
# Make the shared code in src/ importable, same two-hop pattern as the cleaning notebook.
import sys
from pathlib import Path

PROJECT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT / 'src'))

import numpy as np
import pandas as pd

# Method imports, one per outlier method, so the top of the notebook stays the single
# honest list of dependencies.
from sklearn.covariance import MinCovDet           # Method 1: robust Mahalanobis
from sklearn.neighbors import LocalOutlierFactor   # Method 2: LOF (density)
# isotree's IsolationForest IS the Extended Isolation Forest (random hyperplane splits),
# unlike sklearn's axis-aligned one. Alias it so the two never get confused.
from isotree import IsolationForest as ExtendedIsolationForest  # Method 3: EIF

# Load the scaled features written by 01_cleaning_cat2.ipynb.
scaled_path = PROJECT / 'data' / 'processed' / 'phase_1' / 'catalog2_features_scaled.csv'
scaled = pd.read_csv(scaled_path)

# is_repeater round-trips through CSV as the string 'True'/'False', not a bool. Cast it
# back explicitly so later code can use it as a boolean mask.
scaled['is_repeater'] = scaled['is_repeater'].astype(str).str.lower() == 'true'

# Split ID/label columns from the feature columns. X is what the methods see.
id_cols = ['tns_name', 'sub_num', 'is_repeater', 'repeater_name', 'catalog1_flag']
feature_cols = [c for c in scaled.columns if c not in id_cols]
X = scaled[feature_cols].values

print(f'Loaded {len(scaled)} sub-burst rows, {len(feature_cols)} features')
print(f'  repeaters: {int(scaled["is_repeater"].sum())}')
print(f'  one-offs:  {int((~scaled["is_repeater"]).sum())}')
print(f'  features:  {feature_cols}')

Loaded 4109 sub-burst rows, 8 features
  repeaters: 1047
  one-offs:  3062
  features:  ['dm_fitb', 'width_fitb', 'flux', 'fluence', 'sp_idx', 'sp_run', 'peak_freq', 'bandwidth']


## Method 1: Robust Mahalanobis distance (MCD)

**Principle: distance from the centre.** Mahalanobis distance measures how far each
burst sits from the middle of the data cloud, but unlike plain straight-line distance it
accounts for the shape of the cloud. It stretches along directions where the features
spread widely and squeezes along tight, correlated directions. So a burst that is unusual
*given how the features normally move together* scores high, even if no single feature is
extreme on its own.

**The masking problem.** The distance needs a centre and a covariance to measure against.
The obvious choice, the plain mean and covariance of every row, is computed from data
that *includes* the outliers we are hunting. Outliers inflate the covariance and drag the
mean toward themselves, making them look closer to the centre than they really are. They
hide inside a cloud they themselves distorted. This is the masking effect.

**The MCD fix.** Minimum Covariance Determinant finds the most tightly packed subset of
the data (the subset whose covariance matrix has the smallest determinant) and uses
*only* that clean core to define the centre and covariance. Distances are then measured
against that uncontaminated reference, so the outliers can no longer mask themselves.

In [2]:
# MinCovDet finds the tight 'clean core' of the data and estimates a robust mean and
# covariance from it. We leave support_fraction at its default (~half the data), which
# gives the highest robustness to outliers. random_state fixes the internal random
# subsampling so the run is reproducible.
mcd = MinCovDet(random_state=0).fit(X)

# .mahalanobis() returns the SQUARED Mahalanobis distance of each burst from the robust
# centre. Bigger = further from the clean core = more outlying.
maha_score = mcd.mahalanobis(X)

# Build the results table that all three methods will write into. Method 1 seeds it with
# the IDs and labels; Methods 2 and 3 add their own score/rank cols.
results = scaled[['tns_name', 'sub_num', 'is_repeater']].copy()
results['maha_score'] = maha_score
# rank 1 = most outlying. method='first' breaks ties by row order so every row gets a
# unique integer rank.
results['maha_rank'] = results['maha_score'].rank(ascending=False, method='first').astype(int)

# Show the 10 most outlying rows by this method, with the repeater flag, for an early
# feel for whether repeaters are surfacing near the top.
top10 = results.sort_values('maha_rank').head(10).copy()
top10['maha_score'] = top10['maha_score'].round(1)
print('Top 10 by robust Mahalanobis distance:')
print(top10[['maha_rank', 'tns_name', 'sub_num', 'maha_score', 'is_repeater']].to_string(index=False))
print(f'\nRepeaters in this top 10: {int(top10["is_repeater"].sum())} of 10')

Top 10 by robust Mahalanobis distance:
 maha_rank     tns_name  sub_num  maha_score  is_repeater
         1 FRB20190929C        1   1298653.3         True
         2 FRB20200321E        0    913674.5        False
         3 FRB20181119D        1    891715.8        False
         4 FRB20190625E        1    826668.9         True
         5 FRB20201119A        0    706688.3        False
         6 FRB20200124E        0    692379.3        False
         7 FRB20210203C        0    688380.9         True
         8 FRB20190625E        0    614565.8         True
         9 FRB20220714C        1    613880.3         True
        10 FRB20200427A        1    490472.7        False

Repeaters in this top 10: 5 of 10


## Method 2: Local Outlier Factor (LOF), swept over k

**Principle: local density.** Where Mahalanobis asks 'how far is this burst from the
global centre?', LOF asks a more local question: 'is this burst sitting in a sparser
patch than its own neighbours are?' It compares each burst's local density to the density
around its *k* nearest neighbours. A burst packed in among similar-density neighbours
scores around 1 (an inlier); a burst in a relatively empty pocket compared to its
neighbours scores well above 1 (an outlier). This catches a different kind of anomaly: a
burst can be near the global centre and still be locally isolated, which Mahalanobis
would miss.

**Why sweep k.** LOF's entire answer depends on what counts as 'local', set by k, the
neighbourhood size. Too small and the score reacts to noise; too large and 'local' stops
being local. There is no single correct k, so we run a range and look at how stable each
burst's ranking is across the sweep. Cat 1 used {10,20,30,50,75} around sqrt(558)~24.
Cat 2 has ~8x more rows, so we re-bracket around sqrt(N)~67 with {30,50,75,100,150}, so
'local' covers the same fraction of the data it did for Cat 1.

**Reading the sweep.** A burst that stays near the top for every k is a robust local
outlier we can trust. A burst extreme at one k and ordinary at the others is a k-artefact,
not a real anomaly. We summarise each burst by its **median rank across the five k
values** (its consistent standing) and separately count, as a **stability score out of
5**, how many of the sweeps put it in the top 10%.

In [3]:
# The k values to sweep (re-bracketed for Cat 2's larger N; see header), and the size of
# the 'top tier' used for the stability count.
K_VALUES = [30, 50, 75, 100, 150]
N = len(scaled)
top_n = int(round(0.10 * N))  # top 10% of rows

# For each k: fit LOF, turn its score into a 'bigger = more outlying' number, and rank.
# We collect one rank column per k so we can watch each burst move across the sweep.
lof_ranks = pd.DataFrame(index=scaled.index)
for k in K_VALUES:
    lof = LocalOutlierFactor(n_neighbors=k)
    lof.fit(X)
    # negative_outlier_factor_ is the NEGATED LOF: inliers ~ -1, outliers more negative.
    # Flip the sign so bigger = more outlying, matching the Mahalanobis convention.
    score_k = -lof.negative_outlier_factor_
    lof_ranks[k] = pd.Series(score_k, index=scaled.index).rank(ascending=False, method='first').astype(int)

# Summarise the sweep by the MEDIAN rank across the five k values. Median, not mean, so a
# single stray k cannot drag a burst up or down. Lower median = consistently outlying.
results['lof_median_rank'] = lof_ranks.median(axis=1)
# Re-rank the median into a clean 1..N integer so it sits alongside maha_rank.
results['lof_rank'] = results['lof_median_rank'].rank(ascending=True, method='first').astype(int)
# Stability: in how many of the 5 sweeps did this burst land in the top 10%?
results['lof_stability'] = (lof_ranks <= top_n).sum(axis=1)

# Show the 10 most outlying rows by median rank, with their rank at EACH k so the
# stability (or instability) across the sweep is visible directly.
top = results.sort_values('lof_rank').head(10)
detail = lof_ranks.loc[top.index].copy()
detail.columns = [f'k={k}' for k in K_VALUES]
show = pd.concat([scaled.loc[top.index, ['tns_name', 'sub_num', 'is_repeater']], detail,
                  results.loc[top.index, ['lof_stability']]], axis=1)
print('Top 10 by LOF median rank, with rank at each k:')
print(show.to_string(index=False))
print(f'\nRepeaters in this top 10: {int(scaled.loc[top.index, "is_repeater"].sum())} of 10')
print(f'(top tier = top {top_n} rows = top 10%; stability is out of {len(K_VALUES)} sweeps)')

Top 10 by LOF median rank, with rank at each k:
    tns_name  sub_num  is_repeater  k=30  k=50  k=75  k=100  k=150  lof_stability
FRB20190929C        1         True     1     1     1      1      1              5
FRB20200321E        0        False     4     2     2      2      2              5
FRB20181119D        1        False     7     3     3      3      4              5
FRB20210514C        0         True    10     4     5      4      3              5
FRB20190625E        1         True     6     6     4      6      6              5
FRB20210302E        0         True    23     7     6      5      5              5
FRB20190625E        0         True     2     5     7      7      7              5
FRB20201119A        0        False    18     8     8      8      8              5
FRB20200124E        0        False    20     9     9      9      9              5
FRB20210203C        0         True    22    10    10     10     10              5

Repeaters in this top 10: 6 of 10
(top tier = top

## Method 3: Extended Isolation Forest (EIF)

**Principle: ease of isolation.** This asks yet another question: 'how few random cuts
does it take to fence this burst off on its own?' It builds many random binary trees,
each repeatedly slicing the data into smaller regions. A normal burst sits deep in the
crowd and needs many cuts before it ends up alone; an anomaly, being rare and different,
gets isolated in a handful of cuts. Average that path length over a whole forest and you
have an anomaly score: short path = easily isolated = outlier. This needs no notion of
distance or density at all, which is why it can catch things the other two miss.

**Why the *Extended* version.** Standard Isolation Forest (scikit-learn's) only ever cuts
straight across one feature at a time; its splits are axis-aligned. That leaves a known
blind spot: artefact bands of artificially low score running parallel to the axes, and it
struggles with anomalies that are only unusual in a *diagonal* combination of features.
Extended Isolation Forest (Hariri et al. 2019, arXiv:1811.02141) replaces the
axis-aligned cuts with **random hyperplanes**, slicing at random angles through several
features at once, removing that bias while keeping the isolation principle untouched.
This is why we use `isotree` rather than scikit-learn.

**Settings.** Full extension (`ndim` set to all 8 features, so every cut is a hyperplane
through the whole feature space), 100 trees, and a 256-burst subsample per tree (the
classic isolation-forest sampling from Liu et al. 2008). A fixed seed keeps the run
reproducible. The score is already standardised to [0, 1], higher = more easily isolated
= more outlying, matching the other two methods.

In [4]:
# ndim = number of features means every split is a hyperplane through ALL 8 features
# (full extension). ndim=1 would collapse this back to the axis-aligned IF we rejected.
n_features = X.shape[1]
eif = ExtendedIsolationForest(
    ndim=n_features,    # full extension: random hyperplanes, not axis-aligned cuts
    ntrees=100,         # forest size; 100 is the value used in the original IF/EIF papers
    sample_size=256,    # each tree sees a 256-burst random subsample (Liu et al. 2008)
    random_seed=0,      # reproducible run
)
eif.fit(X)
# predict() returns the standardised anomaly score in [0, 1]: higher = isolated in fewer
# cuts = more outlying. Same 'bigger = more outlying' convention as the others.
eif_score = eif.predict(X)

results['eif_score'] = eif_score
# rank 1 = most outlying, same convention as maha_rank and lof_rank.
results['eif_rank'] = results['eif_score'].rank(ascending=False, method='first').astype(int)

# Top 10 by this method, with the repeater flag, same preview as the other two.
top10 = results.sort_values('eif_rank').head(10).copy()
top10['eif_score'] = top10['eif_score'].round(3)
print('Top 10 by Extended Isolation Forest anomaly score:')
print(top10[['eif_rank', 'tns_name', 'sub_num', 'eif_score', 'is_repeater']].to_string(index=False))
print(f'\nRepeaters in this top 10: {int(top10["is_repeater"].sum())} of 10')

/Users/elicox/opt/anaconda3/envs/FRB_Research/lib/python3.11/site-packages/isotree/__init__.py:97: UserWarning: Attempting to use more than 1 thread, but package was built without multi-threading support - see the project's GitHub page for more information.
  warnings.warn(msg_omp)


Top 10 by Extended Isolation Forest anomaly score:
 eif_rank     tns_name  sub_num  eif_score  is_repeater
        1 FRB20190929C        1      0.812         True
        2 FRB20200321E        0      0.794        False
        3 FRB20181119D        1      0.793        False
        4 FRB20210302E        0      0.786         True
        5 FRB20210514C        0      0.780         True
        6 FRB20190625E        1      0.770         True
        7 FRB20211114A        1      0.767         True
        8 FRB20190625E        0      0.761         True
        9 FRB20210203C        0      0.754         True
       10 FRB20200124E        0      0.752        False

Repeaters in this top 10: 7 of 10


In [5]:
# Combine the three methods' per-burst outputs into one table and write it to disk. This
# CSV is the single artefact the validation notebook reads. We persist each method's RAW
# quantity and fix the direction later, in validation, so nothing gets silently
# transformed here where we'd forget about it.
#
# Direction of each column (re-stated in validation, no black box):
#   maha_score        higher = more anomalous
#   eif_score         higher = more anomalous
#   lof_median_rank   LOWER  = more anomalous  (a rank across the k-sweep, not a score)
#
# repeater_name is the repeating source id (blank/NaN for one-offs). Validation needs it
# to group a repeater's many sub-bursts back to one source. catalog1_flag is carried for
# the later instrument-vs-astrophysics triage. 'results' was built in row order and never
# reordered, so scaled's columns line up with it positionally.

scores = results[['tns_name', 'sub_num', 'is_repeater',
                  'maha_score', 'lof_median_rank', 'eif_score']].copy()
scores['repeater_name'] = scaled['repeater_name'].values
scores['catalog1_flag'] = scaled['catalog1_flag'].values
scores = scores[['tns_name', 'sub_num', 'repeater_name', 'catalog1_flag', 'is_repeater',
                 'maha_score', 'lof_median_rank', 'eif_score']]

out_path = PROJECT / 'data' / 'processed' / 'phase_1' / 'catalog2_method_scores.csv'
scores.to_csv(out_path, index=False)

print(f'Wrote {len(scores)} rows to {out_path.relative_to(PROJECT)}')
print(f'Columns: {list(scores.columns)}')
print(f'Repeaters: {int(scores["is_repeater"].sum())}, one-offs: {int((~scores["is_repeater"]).sum())}')
print()
print(scores.head().to_string(index=False))

Wrote 4109 rows to data/processed/phase_1/catalog2_method_scores.csv
Columns: ['tns_name', 'sub_num', 'repeater_name', 'catalog1_flag', 'is_repeater', 'maha_score', 'lof_median_rank', 'eif_score']
Repeaters: 1047, one-offs: 3062

    tns_name  sub_num repeater_name  catalog1_flag  is_repeater  maha_score  lof_median_rank  eif_score
FRB20180904A        0           NaN              1        False    8.609904           1935.0   0.391458
FRB20180906A        0           NaN              1        False   50.198539            286.0   0.471323
FRB20180906B        0           NaN              1        False   13.532037             68.0   0.501055
FRB20180907D        0           NaN              1        False   33.887865           1844.0   0.433480
FRB20180907A        0           NaN              1        False   21.524926           2100.0   0.389409
